# Overton

Database tracking how scholarly research is cited in policy documents — government reports, think tank publications, IGO documents, NGO reports.

**Base URL:** `https://app.overton.io`  
**Two endpoints:** `/articles.php` (scholarly articles) and `/documents.php` (policy documents).

Mental model: Overton ingests **policy documents** from public-sector sources, extracts citations from them, and records each cited **scholarly article**.  An article's `cited_by_documents` field gives the policy citation network.

What it gives the pipeline: per researcher, which of their works have been cited by policy documents, and full metadata on those documents.

## Setup

In [ ]:
import json, os, time, urllib.parse
import requests

try:
    import boto3
    sm = boto3.client('secretsmanager', region_name='us-east-1')
    OV_KEY = os.environ.get('OVERTON_API_KEY') or sm.get_secret_value(SecretId='overton/api-keys/overton')['SecretString']
except Exception:
    OV_KEY = os.environ['OVERTON_API_KEY']

OV_BASE = 'https://app.overton.io'
PSU_ROR = 'https://ror.org/04p491231'

def ov_get(endpoint, **params):
    params['format'] = 'json'
    params['api_key'] = OV_KEY
    r = requests.get(f'{OV_BASE}/{endpoint}', params=params, timeout=30)
    r.raise_for_status()
    time.sleep(0.2)
    return r.json()

def ov_post(endpoint, body):
    # The Content-Type header is REQUIRED. Without it Overton can't parse the body
    # as form data and returns an error response with no 'set' key (KeyError on access).
    r = requests.post(f'{OV_BASE}/{endpoint}',
                      params={'format':'json','api_key':OV_KEY},
                      data=body,
                      headers={'Content-Type': 'application/x-www-form-urlencoded'},
                      timeout=30)
    if not r.ok:
        raise RuntimeError(f'POST {endpoint} -> {r.status_code}: {r.text[:200]}')
    return r.json()

def pretty(o, limit=2000):
    s = json.dumps(o, indent=2, default=str, ensure_ascii=False)
    print(s if len(s) < limit else s[:limit] + f'\n... ({len(s):,} chars total)')

print('ready')


## 1. `/articles.php?query=<ORCID>` — current pipeline pattern

What today's pipeline does: free-text search, jamming the ORCID into the `query` parameter.  Overton matches it against the `orcids` field on each article.

**Caveat:** this only finds articles where Overton has tagged the researcher's ORCID.  Older papers, co-authored work, or papers indexed before the ORCID was registered get missed — typically catching only a fraction of a researcher's actual policy-cited articles.

In [ ]:
with open('../data/pipeline/orcid_webaccess_map.json') as f:
    orcid_map = json.load(f)
sample_orcid = next(iter(orcid_map))

d = ov_get('articles.php', query=sample_orcid, sort='relevance')
print(f'query={sample_orcid}')
print(f'  total_results: {d["query"]["total_results"]}')
print(f'  description:   {d["query"].get("description")}')
print(f'  pages:         {d["query"].get("pages")}')

## 2. Article record — what fields come back

Each article has rich metadata plus a nested `cited_by_documents` array — the policy documents citing this article.

In [ ]:
# results structure is {'results': {'results': [list of articles]}}
articles = d['results']['results']
art = articles[0]
print(f'Article keys: {list(art.keys())}\n')
for k in ['title', 'doi', 'authors', 'orcids', 'affiliations',
         'journal', 'publisher', 'published_on', 'citations',
         'last_cited', 'oa_status']:
    v = art.get(k)
    print(f'  {k:<20} {str(v)[:90]!r}')

In [ ]:
# r_open_institution_authors is the canonical author identifier Overton uses internally
print(f'r_open_institution_authors: {art.get("r_open_institution_authors")}')

In [ ]:
# Nested cited_by_documents — the policy citation network
cited = art.get('cited_by_documents', [])
print(f'{len(cited)} policy documents cite this article\n')
if cited:
    print('First citing document:')
    pretty(cited[0])

## 3. `/articles.php?r_open_institution_authors=...` — canonical author identifier

Overton stores each author under `{ROR}__OVSEP__{Name}__OVSEP__{lowercase_name}`.  Filtering by this identifier finds articles authored by that exact person at that exact institution.

**Caveat:** name spellings have to match Overton's internal record exactly.  Unicode hyphens, accents, and middle-initial periods cause misses.  Prefer ORCID or DOI-set when you have them.

In [ ]:
# Use the canonical identifier we just saw on the article
psu_id = next((v for v in (art.get('r_open_institution_authors') or []) if 'penn' in v.lower() or '04p491231' in v), None)
print(f'Identifier: {psu_id!r}')
if psu_id:
    d = ov_get('articles.php', r_open_institution_authors=psu_id)
    print(f'  total_results: {d["query"]["total_results"]}')
    print(f'  description:   {d["query"].get("description")}')

## 4. `/articles.php?dois=<DOI>` — single DOI lookup
Find the Overton article record for a specific DOI.

In [ ]:
DOI = art.get('doi')
d = ov_get('articles.php', dois=DOI)
print(f'dois={DOI}')
print(f'  total_results: {d["query"]["total_results"]}')
if d['query']['total_results']:
    a = d['results']['results'][0]
    print(f'  title: {a["title"][:70]}')
    print(f'  citations (policy docs): {a["citations"]}')

## 5. `/generate_id_set.php` — batch DOIs into a set

POST endpoint that takes newline-separated DOIs and returns a set ID.  The set ID can then be used as the value of `dois=` (articles) or `plain_dois_cited=` (documents) — letting us query 100s of DOIs in one request instead of 100s of requests.

Body format: form-urlencoded, with DOIs newline-separated as the value of `dois`.

In [ ]:
DOIS = [
    '10.1016/j.jadohealth.2015.09.004',
    '10.1093/ntr/ntu071',
    '10.1080/22221751.2024.2321993',
]
body = urllib.parse.urlencode({'dois': '\n'.join(DOIS)})
r = ov_post('generate_id_set.php', body)
set_id = r['set']
print(f'Created set: {set_id}')

## 6. `/articles.php?dois=<set_id>` — articles in the set

In [ ]:
d = ov_get('articles.php', dois=set_id)
print(f'  total_results: {d["query"]["total_results"]}  (we submitted {len(DOIS)} DOIs)')
for a in d['results']['results']:
    print(f'  - {a["title"][:60]:<60}  citations={a["citations"]}')

## 7. `/documents.php?plain_dois_cited=<set_id>` — policy docs citing the set

Same set ID, different endpoint.  Returns every policy doc that cites *any* DOI in the set.

In [ ]:
d = ov_get('documents.php', plain_dois_cited=set_id)
print(f'  total_results: {d["query"]["total_results"]}')
print(f'  description:   {d["query"].get("description")}')

## 8. Document record — what fields come back

In [ ]:
docs = d['results']['results']
doc = docs[0]
print(f'Document keys: {list(doc.keys())}\n')
pretty(doc)

## 9. `/documents.php?query=<text>` — free-text search across documents
Searches the policy doc full-text and metadata.  Useful for content discovery; less useful for tracking a specific researcher (high false-positive rate on common names).

In [ ]:
d = ov_get('documents.php', query='climate change adaptation')
print(f'total: {d["query"]["total_results"]}')
for doc in (d['results']['results'][:3]):
    src = doc.get('source', {})
    print(f'  {doc["title"][:70]}')
    print(f'    {src.get("title")}  ({src.get("country")})  {doc.get("published_on")}')

## 10. End-to-end harvest for one researcher

Combines the moving parts: pull all DOIs for a researcher from OpenAlex, build a set, query both endpoints.  Compare against the current pipeline pattern (`query=<ORCID>`) to see the coverage difference.

In [ ]:
# Step 1: pull DOIs from OpenAlex
OPENALEX_EMAIL = os.environ.get('OPENALEX_EMAIL', 'overton-pipeline@psu.edu')
dois = []
cursor = '*'
while cursor:
    r = requests.get('https://api.openalex.org/works', params={
        'filter': f'author.orcid:{sample_orcid}',
        'per-page': 200,
        'cursor': cursor,
        'select': 'doi',
    }, headers={'User-Agent': f'mailto:{OPENALEX_EMAIL}'}, timeout=30)
    j = r.json()
    for w in j.get('results', []):
        if w.get('doi'):
            dois.append(w['doi'].replace('https://doi.org/', ''))
    cursor = j.get('meta', {}).get('next_cursor')
    time.sleep(0.1)
print(f'Pulled {len(dois)} DOIs for {sample_orcid}')

In [ ]:
# Step 2: create a set
body = urllib.parse.urlencode({'dois': '\n'.join(dois)})
set_id = ov_post('generate_id_set.php', body)['set']
print(f'Set: {set_id}')

# Step 3: compare DOI-set vs. ORCID-text-search
doi_articles = ov_get('articles.php', dois=set_id)['query']['total_results']
doi_docs     = ov_get('documents.php', plain_dois_cited=set_id)['query']['total_results']
orcid_articles = ov_get('articles.php', query=sample_orcid, sort='relevance')['query']['total_results']
orcid_docs     = ov_get('documents.php', query=sample_orcid)['query']['total_results']

print(f'\n                         Articles  Policy Docs')
print(f'  DOI-set (new method):  {doi_articles:>4}      {doi_docs:>4}')
print(f'  ORCID free-text (now): {orcid_articles:>4}      {orcid_docs:>4}')
print(f'  Coverage gain:         {doi_articles - orcid_articles:+4}     {doi_docs - orcid_docs:+5}')

## What the pipeline pulls today (Stages 3 & 4: Overton)

**Stage 3** loops over each mapped researcher's ORCID:

| Endpoint | Used for |
|---|---|
| `/articles.php?query=<ORCID>` | scholarly articles + nested `cited_by_documents` (the policy citation network) |

**Stage 4** then enriches every unique `policy_document_id` discovered in Stage 3:

| Endpoint | Used for |
|---|---|
| `/documents.php?query=<doc_id>` | full document metadata (source, country, type, topics, SDGs, PDF URL) |

**After the planned refactor** the flow shifts to a DOI-set approach:

| Step | Endpoint | Replaces |
|---|---|---|
| 1 | OpenAlex `/works?filter=author.orcid:X` | (new — pulls full DOI list per researcher) |
| 2 | `POST /generate_id_set.php` (newline-separated DOIs) | (new — batches DOIs into a set) |
| 3 | `/articles.php?dois=<set_id>` | replaces `articles.php?query=<ORCID>` |
| 4 | `/documents.php?plain_dois_cited=<set_id>` | replaces per-doc-id enrichment |